# 07. Divide data to x and y, scaling and splitting
The last preparation steps of the class notebooks: **Divide data to x and y**, **Data scaling (x only)**, **Data Splitting** and **Data balancing** — and why time series need a different splitting rule.

* **Needs:** `work/dataset.csv`. **Makes:** `work/model_data.csv` (used by the LSTM notebooks).

*Simple notebook series of the project 'This Time Is Different?'. Prepared with AI assistance (declared in `notes/ai_use.md`); a study notebook, not dissertation text.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
folder = '/content/drive/MyDrive/DataSets/ThisTimeIsDifferent/'

import warnings
warnings.filterwarnings('ignore')     # hide warning messages

In [2]:
from pandas import read_csv
dataset = read_csv(folder + 'work/dataset.csv', index_col=0, parse_dates=True)
dataset = dataset.loc[:'2023-08-31']        # main sample: January 2003 - August 2023
print(dataset.shape)

(5243, 17)


## Squared returns in logs
Like the target, the squared returns are very skewed, so the LSTM uses their logarithm. Two days had a return of exactly 0, and $\ln(0)$ does not exist; `clip(lower=0.00000001)` replaces anything smaller than $10^{-8}$ by $10^{-8}$ first.

In [3]:
print('days with a zero return:', (dataset['r2'] == 0).sum())
from numpy import log
dataset['log_r2'] = log(dataset['r2'].clip(lower=0.00000001))
dataset['log_us_r2'] = log(dataset['us_r2'].clip(lower=0.00000001))
dataset['log_euro_r2'] = log(dataset['euro_r2'].clip(lower=0.00000001))

days with a zero return: 2


## Divide data to x and y
* **x** — six inputs per day, two for each market: the return (**direction**) and the log squared return (**size**).
* **y** — the target: the log of next week's realised variance.

In [4]:
x = dataset[['ret', 'log_r2', 'us_ret', 'log_us_r2', 'euro_ret', 'log_euro_r2']]
y = dataset['log_rv5_fwd']
print(x.shape)
print(y.shape)

(5243, 6)
(5243,)


## Data Splitting — why the class method does not work here
In class we wrote `train_test_split(x, y, test_size=0.10, random_state=10)`. It **shuffles** the rows. Look at the first test dates:

In [5]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=10)
print(x_test.index[:6])

DatetimeIndex(['2011-01-19', '2011-10-11', '2012-11-22', '2023-04-13',
               '2023-03-14', '2006-05-05'],
              dtype='datetime64[us]', freq=None)


The test days are mixed with the training days: the model would learn from 2009 and be tested on 2008 — it would **know the future**. For employees this does not matter; for time series it does. So we split **by time** with `shuffle=False`:

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, shuffle=False)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)
print('training ends:', x_train.index[-1].date(), '| testing starts:', x_test.index[0].date())

(4194, 6)
(1049, 6)
(4194,)
(1049,)
training ends: 2019-07-22 | testing starts: 2019-07-23


### The walk-forward design used in the study
One split is not enough: the world changes. The study uses a **walk-forward**: train on all years up to 31 December, forecast the whole next year, then add that year to the training data and repeat — 17 times, for the test years 2007 to 2023.

In [7]:
for year in range(2007, 2024):
    train = dataset.loc[:str(year - 1) + '-12-31']
    test = dataset.loc[str(year) + '-01-01':str(year) + '-12-31']
    print('test year', year, '| training days:', len(train), '| test days:', len(test))

test year 2007 | training days: 1013 | test days: 254
test year 2008 | training days: 1267 | test days: 254
test year 2009 | training days: 1521 | test days: 251
test year 2010 | training days: 1772 | test days: 254
test year 2011 | training days: 2026 | test days: 253
test year 2012 | training days: 2279 | test days: 254
test year 2013 | training days: 2533 | test days: 254
test year 2014 | training days: 2787 | test days: 254
test year 2015 | training days: 3041 | test days: 254
test year 2016 | training days: 3295 | test days: 254
test year 2017 | training days: 3549 | test days: 253
test year 2018 | training days: 3802 | test days: 253
test year 2019 | training days: 4055 | test days: 253
test year 2020 | training days: 4308 | test days: 256
test year 2021 | training days: 4564 | test days: 255
test year 2022 | training days: 4819 | test days: 254
test year 2023 | training days: 5073 | test days: 170


## Data scaling (x only) — fitted on the training years
In class: `x_scaled = StandardScaler().fit_transform(x)` on **all** of x before splitting. With time series this leaks the future: the mean and standard deviation would include the crisis years. Rule: **fit the scaler on the training years only**, then use it on the test year.

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(x.loc[:'2006-12-31'])         # learn mean and std from 2003-2006 only
from pandas import DataFrame
x_scaled = DataFrame(scaler.transform(x), index=x.index, columns=x.columns)
x_scaled.loc[:'2006'].describe().round(2)

,ret,log_r2,us_ret,log_us_r2,euro_ret,log_euro_r2
count,1013.00,1013.00,1013.00,1013.00,1013.00,1013.00
mean,-0.00,0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00
min,-7.48,-3.26,-4.66,-3.30,-5.08,-3.21
25%,-0.54,-0.49,-0.58,-0.60,-0.43,-0.56
50%,0.02,0.19,0.04,0.17,0.03,0.14
75%,0.56,0.71,0.56,0.74,0.49,0.76
max,4.90,2.48,4.39,2.06,5.55,2.15


In the training years the mean is 0 and the standard deviation 1. In the crisis year 2008 the same scale gives much bigger numbers — exactly the information the model should see:

In [9]:
x_scaled.loc['2008'].describe().round(2)

,ret,log_r2,us_ret,log_us_r2,euro_ret,log_euro_r2
count,254.00,254.00,254.00,254.00,254.00,254.00
mean,-0.62,1.08,-0.32,0.84,-0.21,0.44
std,3.62,1.12,3.31,1.16,1.89,1.01
min,-16.92,-3.26,-12.19,-3.30,-5.95,-3.21
25%,-2.44,0.59,-1.75,0.20,-1.06,-0.07
50%,-0.56,1.30,-0.09,0.99,-0.17,0.60
75%,1.02,1.86,0.94,1.66,0.66,1.10
max,11.62,3.21,13.96,3.08,8.50,2.50


## Data balancing
In class, `SMOTE` made extra examples of the rare class (employees who left). Our target is a **number** (regression), not a class, so there is nothing to balance. Crisis weeks are rare, but inventing crisis days would break the time order. Instead, the results are reported **separately for each crisis window**.

In [10]:
dataset.to_csv(folder + 'work/model_data.csv')
print('saved')

saved
